In [0]:
from pyspark.sql import functions as F

regions = (
    spark.table("workspace.bank_lakehouse.bronze_regions")
    .select(
        F.col("shape_iso").alias("region_iso"),
        F.col("shape_name").alias("region_name"),
        F.expr("st_geomfromgeojson(geometry_geojson)").alias("region_geom"),
    )
)

agencies = spark.table("workspace.bank_lakehouse.silver_osm_banks") \
    .select("osm_key", "lat", "lon", "bank_name")

joined = agencies.join(
    F.broadcast(regions),
    F.expr("st_contains(region_geom, st_point(lon, lat, 4326))"),
    "left",
)

print("Nombre de régions trouvées par agence :")
joined.groupBy("osm_key").agg(F.count("region_iso").alias("n_regions")) \
    .groupBy("n_regions").count().orderBy("n_regions").show()

display(joined.groupBy("region_iso", "region_name").count().orderBy(F.desc("count")))

In [0]:
from pyspark.sql import functions as F

expected = [
    "banque", "banque participative", "banque centrale", "transfert d'argent",
    "crédit à la consommation", "crédit immobilier", "microcrédit",
    "change", "assurance", "non identifié",
]

display(
    spark.table("workspace.bank_gold.dim_bank")
    .groupBy("institution_type").count()
    .withColumn("dans_la_liste", F.col("institution_type").isin(expected))
    .withColumn("longueur", F.length("institution_type"))
    .withColumn("hex", F.hex("institution_type"))
    .orderBy("dans_la_liste")
)